In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets huggingface_hub

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-drqngl68/unsloth_3ab909c318ae456c9c68fe71990ccf67
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-drqngl68/unsloth_3ab909c318ae456c9c68fe71990ccf67
  Resolved https://github.com/unslothai/unsloth.git to commit e62085a3d6a77ffbc0ae796836dd55e36769195c
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.9/402.9 kB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 127.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 22.6 MB/s eta 0:00:00

In [2]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",  # ← yeh try karo
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

print(f"Model load ho gaya! Parameters: {model.num_parameters()/1e9:.2f}B")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.15: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


Model load ho gaya! Parameters: 3.21B


In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj",
        "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

model.print_trainable_parameters()
print("LoRA adapter lag gaya!")

Unsloth 2026.3.15 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511
LoRA adapter lag gaya!


In [4]:
from datasets import load_dataset

SYSTEM_PROMPT = """You are a highly knowledgeable medical assistant.
Answer clinical questions accurately and clearly."""

def format_prompt(examples):
    texts = []
    for instruction, output in zip(examples["instruction"], examples["output"]):
        text = tokenizer.apply_chat_template(
            [
                {"role": "system",    "content": SYSTEM_PROMPT},
                {"role": "user",      "content": instruction},
                {"role": "assistant", "content": output},
            ],
            tokenize = False,
            add_generation_prompt = False,
        )
        texts.append(text)
    return {"text": texts}

# Dataset load karo
dataset = load_dataset("medalpaca/medical_meadow_medqa", split="train")

# Format karo
dataset = dataset.map(format_prompt, batched=True)

# Sirf 2000 examples use karo (jaldi training ke liye)
dataset = dataset.select(range(2000))

print(f"Dataset ready! Total examples: {len(dataset)}")
print("\nEk sample dekho:")
print(dataset[0]["text"][:300])

README.md: 0.00B [00:00, ?B/s]

medical_meadow_medqa.json:   0%|          | 0.00/10.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10178 [00:00<?, ? examples/s]

Map:   0%|          | 0/10178 [00:00<?, ? examples/s]

Dataset ready! Total examples: 2000

Ek sample dekho:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 27 Mar 2026

You are a highly knowledgeable medical assistant.
Answer clinical questions accurately and clearly.<|eot_id|><|start_header_id|>user<|end_header_id|>

Please answer with one of


In [5]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# Trainer configure karo
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 50,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 42,
        output_dir = "./medical_output",
        report_to = "none",
    ),
)

# VRAM check karo pehle
gpu_stats = torch.cuda.get_device_properties(0)
total_vram = round(gpu_stats.total_memory / 1024**3, 1)
print(f"GPU: {gpu_stats.name}")
print(f"Total VRAM: {total_vram} GB")
print("\nTraining shuru ho rahi hai... thoda wait karo ☕")

# TRAIN!
trainer_stats = trainer.train()

print(f"\n✅ Training complete!")
print(f"Train Loss: {trainer_stats.metrics['train_loss']:.4f}")
print(f"Time laga: {trainer_stats.metrics['train_runtime']/60:.1f} minutes")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/2000 [00:00<?, ? examples/s]

GPU: Tesla T4
Total VRAM: 14.6 GB

Training shuru ho rahi hai... thoda wait karo ☕


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,000 | Num Epochs = 3 | Total steps = 750
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Step,Training Loss
10,4.807828
20,3.274696
30,1.286396
40,0.697293
50,0.602196
60,0.468563
70,0.348761
80,0.352905
90,0.314966
100,0.301682



✅ Training complete!
Train Loss: 0.3984
Time laga: 18.0 minutes


In [6]:
# Save model locally in Colab storage
model.save_pretrained("medical_qlora_adapter")
tokenizer.save_pretrained("medical_qlora_adapter")
print("✅ Model saved successfully — ./medical_qlora_adapter/ folder")

✅ Model saved successfully — ./medical_qlora_adapter/ folder


In [7]:
import torch
import gc
from unsloth import FastLanguageModel
from transformers import AutoTokenizer

# ── Step 1: Memory clear ─────────────────────────
print("Clearing memory...")
for var in ["model", "trainer", "model_inf", "base_model"]:
    try:
        del globals()[var]
    except: pass

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

print(f"✅ Memory cleared!")
print(f"Free VRAM: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

# ── Step 2: Load fine-tuned model ─────────────────────────
print("\nLoading fine-tuned model and tokenizer...")
model_inf, tokenizer_inf = FastLanguageModel.from_pretrained(
    model_name="medical_qlora_adapter",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True
)
FastLanguageModel.for_inference(model_inf)

if tokenizer_inf.pad_token is None:
    tokenizer_inf.pad_token = tokenizer_inf.eos_token

print("✅ Model ready!")
print(f"VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# ── Step 3: System prompt & function ─────────────────────
SYSTEM_PROMPT = "You are a highly knowledgeable medical assistant. Answer clinical questions accurately and clearly."

def ask_medical_question(question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    text = tokenizer_inf.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer_inf(text, return_tensors="pt", padding=True).to("cuda")
    input_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        outputs = model_inf.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=512,  # lamba answer ke liye badha diya
            do_sample=False,
            pad_token_id=tokenizer_inf.eos_token_id,
            use_cache=False,
        )

    answer = tokenizer_inf.decode(outputs[0][input_len:], skip_special_tokens=True)
    return answer.strip()

# ── Step 4: Chatbot loop ─────────────────────────────
print("\n🩺 Medical Chatbot Ready! Type 'exit' to quit.\n")
while True:
    user_input = input("You: ")
    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye! Take care 🩺")
        break
    response = ask_medical_question(user_input)
    print(f"Chatbot: {response}\n")

Clearing memory...
✅ Memory cleared!
Free VRAM: 14.56 GB

Loading fine-tuned model and tokenizer...
==((====))==  Unsloth 2026.3.15: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


✅ Model ready!
VRAM used: 3.34 GB

🩺 Medical Chatbot Ready! Type 'exit' to quit.

You: What are the early signs of a heart attack?


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.1

Chatbot: The early signs of a heart attack can be subtle and may not always be immediately recognized. However, here are some common symptoms to look out for:

1. Chest pain or discomfort: This is the most common symptom of a heart attack. The pain may feel like pressure, tightness, or a squeezing sensation in the center of the chest that may radiate to the arms, back, neck, jaw, or stomach.
2. Shortness of breath: You may feel like you can't catch your breath or that you're having trouble breathing.
3. Pain or discomfort in the arms, back, neck, jaw, or stomach: In addition to chest pain, you may experience pain or discomfort in other areas of your body.
4. Cold sweats: You may feel cold, even in a warm environment.
5. Lightheadedness or dizziness: You may feel like you're going to pass out.
6. Fatigue: You may feel extremely tired or weak.
7. Nausea and vomiting: You may feel queasy or vomit.
8. Rapid or irregular heartbeat: Your heart may beat too quickly or irregularly.
9. Palpitat

Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Chatbot: Ibuprofen and paracetamol (also known as acetaminophen in some countries) are two commonly used over-the-counter pain relievers. The main difference between them is their mechanism of action and side effect profile.

Ibuprofen is a nonsteroidal anti-inflammatory drug (NSAID) that works by inhibiting the production of prostaglandins, which are hormone-like substances that cause pain and inflammation. It also has anti-inflammatory and antipyretic (fever-reducing) effects.

Paracetamol, on the other hand, is a non-opioid analgesic that works by blocking the production of prostaglandins in the brain, which relieves pain. It does not have anti-inflammatory effects.

In terms of side effects, ibuprofen can cause gastrointestinal problems such as stomach ulcers and bleeding, especially when taken for extended periods. Paracetamol is generally considered safer, but it can cause liver damage if taken in high doses.

In summary, ibuprofen is a more potent anti-inflammatory agent, while 

Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Chatbot: Paracetamol, also known as acetaminophen, is a widely used over-the-counter analgesic and antipyretic medication. Its mechanism of action is not fully understood, but it is believed to work through several pathways:

1. Inhibition of prostaglandin synthesis: Paracetamol inhibits the enzyme cyclooxygenase (COX), which is involved in the production of prostaglandins. Prostaglandins are mediators of inflammation and pain.
2. Inhibition of phospholipase A2: Paracetamol also inhibits the enzyme phospholipase A2, which is involved in the production of prostaglandins.
3. Blockade of TRPV1 receptors: Paracetamol may block the transient receptor potential vanilloid 1 (TRPV1) receptor, which is involved in the transmission of pain signals.
4. Inhibition of the release of neurotransmitters: Paracetamol may inhibit the release of neurotransmitters such as substance P, which is involved in the transmission of pain signals.

Overall, the exact mechanism of action of paracetamol is not fully